# Viral Agent - Run Full Pipeline in Colab

This notebook runs your project scripts in order:

1. Mount Google Drive
2. Install dependencies
3. Configure project paths and API keys
4. Check metadata/videos exist
5. Run `02_feature_extraction.py`
6. Run `03_merge_dataset.py`
7. Run `04_train_model.py`
8. Optionally run `05_agent_inference.py`

Upload this notebook together with your `.py` scripts, or place the scripts in the Drive folder configured below.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install dependencies
# This can take several minutes. Use a GPU runtime for CLIP/Whisper/MLP.
!pip install -q \
  yt-dlp google-api-python-client pandas numpy pyarrow tqdm \
  scikit-learn xgboost lightgbm optuna \
  torch torchvision torchaudio transformers pillow opencv-python \
  openai-whisper librosa soundfile sentence-transformers groq

In [ ]:
# 3. Configure paths and secrets
import os
from pathlib import Path

# Main data/model folder used by config.py.
# Keep this as your existing Google Drive project folder unless you moved the data.
BASE = '/content/drive/MyDrive/viral_agent'
os.environ['VIRAL_AGENT_BASE'] = BASE

# Optional: required only for LLM feature extraction / LLM inference.
# Prefer setting it in Colab Secrets, then uncomment this if needed:
# from google.colab import userdata
# os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY') or ''

# Folder where the .py scripts are located.
# Option A: upload scripts to /content, then leave this as /content.
# Option B: put scripts in Drive and set PROJECT_DIR to that folder.
PROJECT_DIR = Path('/content')

print('BASE:', BASE)
print('PROJECT_DIR:', PROJECT_DIR)

In [ ]:
# 4. If your scripts are in Drive, copy them into /content for execution.
# Edit DRIVE_SCRIPT_DIR if needed, then run this cell.
DRIVE_SCRIPT_DIR = Path('/content/drive/MyDrive/viral_agent/scripts')

required_scripts = [
    'config.py',
    '02_feature_extraction.py',
    '03_merge_dataset.py',
    '04_train_model.py',
    '05_agent_inference.py',
]

missing = [name for name in required_scripts if not (PROJECT_DIR / name).exists()]
if missing and DRIVE_SCRIPT_DIR.exists():
    for name in required_scripts:
        src = DRIVE_SCRIPT_DIR / name
        if src.exists():
            !cp "{src}" /content/
    PROJECT_DIR = Path('/content')

missing = [name for name in required_scripts if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Missing scripts: ' + ', '.join(missing) + '. Upload them to /content or edit DRIVE_SCRIPT_DIR.')

print('All required scripts found.')

## Optional: Data Collection

Your `copy_of_youtube_shorts_ (1).py` and `data_tiktok.py` files are Colab notebook exports. They include Colab cells and hardcoded collection choices, so they are better run manually as notebooks or cleaned before automation.

Before continuing, make sure you have:

- metadata CSV at `/content/drive/MyDrive/viral_agent/datasets/metadata/all_metadata.csv`
- videos referenced by `local_video_path` in that metadata

Important: rotate/remove any hardcoded YouTube API key before sharing those files.

In [ ]:
# 5. Check expected input files/folders
import pandas as pd

metadata_path = Path(BASE) / 'datasets' / 'metadata' / 'all_metadata.csv'
video_dir = Path(BASE) / 'datasets' / 'videos'

print('Metadata:', metadata_path, 'exists=', metadata_path.exists())
print('Video dir:', video_dir, 'exists=', video_dir.exists())

if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    print('Rows:', len(df))
    print('Columns:', list(df.columns))
    display(df.head())
else:
    raise FileNotFoundError(f'Metadata file not found: {metadata_path}')

In [ ]:
# 6. Run feature extraction
# For a first quick test, run only one or two steps, for example:
# !python /content/02_feature_extraction.py --steps clip whisper

!python /content/02_feature_extraction.py

In [ ]:
# 7. Merge extracted features into the final training dataset
!python /content/03_merge_dataset.py

In [ ]:
# 8. Train the model
# Options: xgb, lgbm, rf, mlp
MODEL_TYPE = 'xgb'

!python /content/04_train_model.py --model {MODEL_TYPE}

In [ ]:
# 9. Optional inference on a new local video
# Set VIDEO_PATH to a file in Drive or /content.
# Leave RUN_INFERENCE = False if you only want to train.

RUN_INFERENCE = False
VIDEO_PATH = '/content/drive/MyDrive/viral_agent/test_video.mp4'
VIDEO_TITLE = ''
PLATFORM = 'unknown'  # youtube, tiktok, instagram, unknown

if RUN_INFERENCE:
    !python /content/05_agent_inference.py --video "{VIDEO_PATH}" --title "{VIDEO_TITLE}" --platform {PLATFORM} --output "{BASE}/results/inference_report.json"
else:
    print('Skipping inference. Set RUN_INFERENCE = True to run it.')

In [ ]:
# 10. Show outputs
from pathlib import Path

for rel in [
    'features/final_model_dataset.parquet',
    'features/feature_names.json',
    'models/best_model.pkl',
    'models/model_meta.json',
    'results/cv_scores.json',
]:
    p = Path(BASE) / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)